# Tenuto

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kyleconciso/tenuto/blob/main/notebooks/colab_training.ipynb)

Predicts human performance nuance (rubato, micro-timing, velocity, articulation, sustain pedal) from sheet music or MIDI scores.

### Stage 1: Setup Repository, Mount Google Drive & Environment
**Note:** To enable GPU acceleration in Colab, go to **Runtime > Change runtime type > T4 GPU**.

In [ ]:
# 1. Clone repo if needed
import os
if not os.path.exists('/content/tenuto'):
    !git clone https://github.com/kyleconciso/tenuto.git /content/tenuto

# 2. Mount Google Drive for automatic checkpoint saving & loading (Idempotent)
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception:
    pass

# 3. Set working directory & PYTHONPATH globally
%cd /content/tenuto
%env PYTHONPATH=/content/tenuto:.

# 4. Verify PyTorch GPU & install dependencies
import torch
print("Current Directory:", os.getcwd())
print("PyTorch:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
else:
    print("⚠️ Running on CPU. Switch to GPU in Colab via: Runtime > Change runtime type > T4 GPU")

!pip install -q partitura mido scipy tqdm matplotlib huggingface_hub midi2audio pandas pyarrow

### Stage 2: Download Dataset

In [ ]:
%cd /content/tenuto
!PYTHONPATH=/content/tenuto python3 -m src.download_dataset --dataset combined --pianocore_subset PianoCoRe-A

### Stage 3: Preprocess Dataset (40D Note Feature Tensors)

In [ ]:
%cd /content/tenuto
!PYTHONPATH=/content/tenuto python3 -m src.preprocess --data_dir ./data --processed_dir ./data/processed

### Stage 4: Train Transformer Backbone (Auto-Saves to Google Drive)

In [ ]:
%cd /content/tenuto
!PYTHONPATH=/content/tenuto python3 -m src.train --model_type transformer --in_features 40 --epochs 20 --batch_size 16 --lr 0.0001

### Stage 5: Expressive Inference

In [ ]:
%cd /content/tenuto
!PYTHONPATH=/content/tenuto python3 -m src.infer --score data/asap/Balakirev/Islamey/xml_score.musicxml --checkpoint checkpoints/best_transformer_model.pth --model_type transformer --output_midi output_expressive.mid

### Stage 6: Listenable Audio Comparison 🎧

In [ ]:
%cd /content/tenuto
!apt-get -qq update && apt-get -qq install -y fluidsynth fluid-soundfont-gm timidity

import sys
sys.path.insert(0, "/content/tenuto")
from src.audio import play_audio_in_colab

print("🎵 1. Playing Original Flat Score (Mechanical):")
play_audio_in_colab("data/asap/Balakirev/Islamey/midi_score.mid", title="Original Flat Score")

print("\n🎵 2. Playing Original Human Performance (Ground Truth):")
play_audio_in_colab("data/asap/Balakirev/Islamey/CHEN04.mid", title="Human Performance")

print("\n🎵 3. Playing Tenuto AI Generated Performance:")
play_audio_in_colab("output_expressive.mid", title="Tenuto AI Expressive Performance")
